## Instalación de Dependencias
Ejecuta esta celda para instalar las librerías necesarias.

# Editor Node Implementation

Este notebook implementa el nodo "Editor" para el sistema de LangGraph especializado en música.
Incluye la definición del estado, herramientas para generar PDF y enviar correos, y la lógica del nodo.

In [1]:
# Importaciones originales del main.ipynb
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr

from langchain_cohere import ChatCohere
from pydantic import BaseModel
import random
from langchain_core.tools import Tool, StructuredTool
from langchain_community.utilities import GoogleSerperAPIWrapper

# Nuevas importaciones para el Editor Node
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from fpdf import FPDF
from typing import TypedDict, List, Dict, Any, Optional
import os

# Cargar variables de entorno
load_dotenv(override=True)

ModuleNotFoundError: No module named 'langgraph'

## 1. Definición del Estado (State)
Definimos la estructura `State` para manejar la información que fluye a través del grafo.

In [ ]:
class State(TypedDict):
    concerts: List[Dict[str, Any]]  # Lista de conciertos
    suno_audio_url: str             # URL del audio generado por Suno
    formato_entrega: str            # 'email' o 'local'
    pdf_path: str                   # Ruta donde se guardó el PDF
    user_email: Optional[str]       # Email del usuario (opcional si es local)
    # Otros campos que puedan existir en el grafo global...
    messages: Annotated[list, add_messages]

## 2. Herramienta PDF (FPDF2)
Función para generar el PDF con el diseño requerido.

In [ ]:
def generate_pdf(concerts: List[Dict[str, Any]], suno_audio_url: str) -> str:
    """
    Genera un PDF con la lista de conciertos y el enlace a Suno.
    Retorna la ruta absoluta del archivo generado.
    """
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    # Título
    pdf.set_font("Arial", style="B", size=16)
    pdf.cell(200, 10, txt="Resumen de Conciertos & Música", ln=True, align='C')
    pdf.ln(10)

    # Tabla de Conciertos
    pdf.set_font("Arial", style="B", size=12)
    # Encabezados
    pdf.cell(40, 10, "Fecha", 1)
    pdf.cell(60, 10, "Ciudad", 1)
    pdf.cell(30, 10, "Precio", 1)
    pdf.cell(60, 10, "Link", 1)
    pdf.ln()

    pdf.set_font("Arial", size=11)
    for concert in concerts:
        # Aseguramos que los datos sean strings
        fecha = str(concert.get("fecha", "N/A"))
        ciudad = str(concert.get("ciudad", "N/A"))
        precio = str(concert.get("precio", "N/A"))
        link = str(concert.get("link", "N/A"))

        pdf.cell(40, 10, fecha, 1)
        pdf.cell(60, 10, ciudad, 1)
        pdf.cell(30, 10, precio, 1)
        # Podríamos hacer un link clicable si FPDF2 lo permite facilemente con cell, 
        # pero por simplicidad mostramos el texto acortado
        pdf.cell(60, 10, link[:25]+"..." if len(link)>25 else link, 1)
        pdf.ln()

    pdf.ln(20)

    # Audio Suno
    pdf.set_font("Arial", style="B", size=12)
    pdf.cell(200, 10, txt="Tu Canción Personalizada (Suno)", ln=True, align='L')
    pdf.set_font("Arial", size=12)
    pdf.write(5, f"Escucha tu canción aquí: {suno_audio_url}", link=suno_audio_url)
    pdf.ln(10)
    
    # Podríamos añadir una imagen QR aquí si tuvieramos la librería qrcode, 
    # pero el requerimiento dice 'o enlace clicable'.

    output_filename = "conciertos_suno.pdf"
    # Guardar en el directorio actual
    pdf.output(output_filename)
    return os.path.abspath(output_filename)

# Envolver en Tool si queremos usarlo como tal, 
# o simplemente llamar a la función desde el nodo.
# El requerimiento pide implementarlo compatible con LangGraph y usar decoradores @tool si es necesario,
# pero dado que es una función interna del nodo Editor, la llamaremos directamente o la definiremos como tool.
# Para cumplir 'utilizando decoradores @tool', la defino como tal:

from langchain_core.tools import tool

@tool
def generate_pdf_tool(concerts: List[Dict[str, Any]], suno_audio_url: str) -> str:
    """Genera el PDF con conciertos y link de audio."""
    return generate_pdf(concerts, suno_audio_url)

## 3. Herramienta SMTP (Email)
Función para enviar el correo si el usuario lo solicita.

In [ ]:
@tool
def send_email_tool(pdf_path: str, receiver_email: str):
    """Envía el PDF generado por correo electrónico."""
    sender_email = os.getenv("EMAIL_SENDER")
    sender_password = os.getenv("EMAIL_PASSWORD")
    
    if not sender_email or not sender_password:
        print("Advertencia: No se han configurado EMAIL_SENDER o EMAIL_PASSWORD. No se envió el correo.")
        return "Error: Credenciales faltantes"

    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = "Tu Resumen de Conciertos y Música"

    body = "Adjunto encontrarás el PDF con la información de los conciertos y tu canción de Suno."
    msg.attach(MIMEText(body, 'plain'))

    # Adjuntar PDF
    try:
        filename = os.path.basename(pdf_path)
        with open(pdf_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())
        
        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename= {filename}",
        )
        msg.attach(part)

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.send_message(msg)
        server.quit()
        return "Correo enviado con éxito"
    except Exception as e:
        print(f"Error enviando correo: {e}")
        return f"Error enviando correo: {e}"

## 4. Lógica del Nodo Editor
El nodo que orquesta la generación y el envío.

In [ ]:
def editor_node(state: State) -> State:
    """
    Nodo Editor: Genera PDF y gestiona el envío.
    """
    print("--- NODO EDITOR ---")
    
    # 1. Extraer datos
    concerts = state.get("concerts", [])
    suno_url = state.get("suno_audio_url", "")
    formato = state.get("formato_entrega", "local")
    email = state.get("user_email", "")

    # 2. Generar PDF
    # Llamamos a la lógica directamente o invocamos la tool si fuera un agente LLM.
    # Como es un nodo determinista, llamamos directo a la función subyacente o a la tool.
    pdf_file = generate_pdf(concerts, suno_url)
    
    # Actualizamos el path en el estado (temporalmente antes de retornar)
    state["pdf_path"] = pdf_file

    # 3. Evaluar condición de envío
    if formato == "email":
        if email:
            print(f"Enviando correo a {email}...")
            # Mock de envío o envío real
            result = send_email_tool.invoke({"pdf_path": pdf_file, "receiver_email": email})
            print(f"Resultado envío: {result}")
        else:
            print("Se solicitó email pero no se proporcionó dirección. Guardado local.")
            print(f"Archivo guardado en local: {pdf_file}")
    else:
        print(f"Archivo guardado en local: {pdf_file}")

    return state

## 5. Verificación (Mock Run)
Probamos el nodo con datos falsos para verificar que funciona.

In [ ]:
# Datos de prueba
mock_state = {
    "concerts": [
        {"fecha": "2024-05-20", "ciudad": "Madrid", "precio": "50 EUR", "link": "http://ticket.com/1"},
        {"fecha": "2024-06-15", "ciudad": "Barcelona", "precio": "45 EUR", "link": "http://ticket.com/2"}
    ],
    "suno_audio_url": "https://suno.ai/song_123",
    "formato_entrega": "local", # Cambiar a 'email' para probar envío (requiere credenciales .env)
    "pdf_path": "",
    "user_email": "test@example.com",
    "messages": []
}

# Ejecutar nodo
final_state = editor_node(mock_state)

print("\nEstado Final:", final_state)